In [1]:
"""
04. Baseline
BoW + MLP

Purpose:
    Establish a simple text classification baseline
    before introducing Transformer architectures.
"""


import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

from sklearn.feature_extraction.text import CountVectorizer
from torch.utils.data import Dataset, DataLoader

from utils.config import (
    SEED,
    NUM_CLASSES,
    BATCH_SIZE,
    EPOCHS,
    LEARNING_RATE,
    MAX_FEATURES,
    BOW_HIDDEN_DIM,
    NUM_WORKERS,
    device,
    MODEL_DIR,
    CONFUSION_DIR
)

from utils.metrics import (
    calculate_metrics,
    get_classification_report,
    get_confusion_matrix,
    print_metrics
)

from utils.training import (
    train_model,
    evaluate
)


# =========================================================
# Reproducibility
# =========================================================

torch.manual_seed(SEED)

if torch.cuda.is_available():

    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


# =========================================================
# Paths
# =========================================================

TRAIN_PATH = Path(
    "results/splits/train.csv"
)

VAL_PATH = Path(
    "results/splits/validation.csv"
)

TEST_PATH = Path(
    "results/splits/test.csv"
)

VECTORIZER_PATH = (
    MODEL_DIR / "bow_vectorizer.pkl"
)

MODEL_PATH = (
    MODEL_DIR / "bow_mlp.pt"
)

CONFUSION_PATH = (
    CONFUSION_DIR / "bow_mlp.png"
)


# =========================================================
# Header
# =========================================================

print("=" * 70)
print("04. Baseline - BoW + MLP")
print("=" * 70)

print()

print(
    f"Device         : {device}"
)

print(
    f"Max Features   : {MAX_FEATURES}"
)

print(
    f"Hidden Dim     : {BOW_HIDDEN_DIM}"
)

print(
    f"Batch Size     : {BATCH_SIZE}"
)

print(
    f"Epochs         : {EPOCHS}"
)

print(
    f"Learning Rate  : {LEARNING_RATE}"
)

print()


# =========================================================
# 1. Load Split Data
# =========================================================

print("[1] Load Split Data")
print("-" * 70)

import pandas as pd

train_df = pd.read_csv(
    TRAIN_PATH
)

val_df = pd.read_csv(
    VAL_PATH
)

test_df = pd.read_csv(
    TEST_PATH
)

print(
    f"Train      : {len(train_df):,}"
)

print(
    f"Validation : {len(val_df):,}"
)

print(
    f"Test       : {len(test_df):,}"
)

print()


# =========================================================
# 2. Extract Text / Labels
# =========================================================

train_texts = (
    train_df["text"]
    .astype(str)
    .tolist()
)

val_texts = (
    val_df["text"]
    .astype(str)
    .tolist()
)

test_texts = (
    test_df["text"]
    .astype(str)
    .tolist()
)

y_train = (
    train_df["label"]
    .astype(int)
    .to_numpy()
)

y_val = (
    val_df["label"]
    .astype(int)
    .to_numpy()
)

y_test = (
    test_df["label"]
    .astype(int)
    .to_numpy()
)


# =========================================================
# 3. Create BoW Vocabulary
# =========================================================

print("[2] Create Bag-of-Words Representation")
print("-" * 70)

vectorizer = CountVectorizer(
    max_features=MAX_FEATURES,
    lowercase=True
)


# ---------------------------------------------------------
# IMPORTANT:
# fit_transform ONLY on training data
# ---------------------------------------------------------

X_train = vectorizer.fit_transform(
    train_texts
)

X_val = vectorizer.transform(
    val_texts
)

X_test = vectorizer.transform(
    test_texts
)


print(
    f"Vocabulary Size : "
    f"{len(vectorizer.vocabulary_):,}"
)

print(
    f"Train Matrix    : "
    f"{X_train.shape}"
)

print(
    f"Validation      : "
    f"{X_val.shape}"
)

print(
    f"Test Matrix     : "
    f"{X_test.shape}"
)

print(
    f"Train Sparsity  : "
    f"{1.0 - X_train.nnz / np.prod(X_train.shape):.4f}"
)

print()


# =========================================================
# 4. Save Vectorizer
# =========================================================

with open(
    VECTORIZER_PATH,
    "wb"
) as f:

    pickle.dump(
        vectorizer,
        f
    )

print(
    f"Vectorizer saved : "
    f"{VECTORIZER_PATH}"
)

print()


# =========================================================
# 5. Dataset
# =========================================================

class SparseBoWDataset(Dataset):

    """
    PyTorch Dataset for scipy sparse BoW matrix.

    Dense conversion is performed only for each sample
    when it is fetched by DataLoader.

    This prevents the entire dataset from being converted
    into a large dense matrix at once.
    """

    def __init__(
        self,
        X,
        y
    ):

        self.X = X
        self.y = torch.tensor(
            y,
            dtype=torch.long
        )

    def __len__(self):

        return self.X.shape[0]

    def __getitem__(
        self,
        index
    ):

        x = self.X[
            index
        ].toarray().ravel()

        x = torch.tensor(
            x,
            dtype=torch.float32
        )

        y = self.y[index]

        return x, y


# =========================================================
# 6. Create Dataset
# =========================================================

train_dataset = SparseBoWDataset(
    X_train,
    y_train
)

val_dataset = SparseBoWDataset(
    X_val,
    y_val
)

test_dataset = SparseBoWDataset(
    X_test,
    y_test
)


# =========================================================
# 7. Create DataLoader
# =========================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS
)


# =========================================================
# 8. Define MLP
# =========================================================

class BoWMLP(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        num_classes
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim
            ),

            nn.ReLU(),

            nn.Linear(
                hidden_dim,
                num_classes
            )
        )

    def forward(
        self,
        x
    ):

        return self.network(x)


# =========================================================
# 9. Create Model
# =========================================================

input_dim = X_train.shape[1]

model = BoWMLP(
    input_dim=input_dim,
    hidden_dim=BOW_HIDDEN_DIM,
    num_classes=NUM_CLASSES
)

model = model.to(device)


print("[3] Model")
print("-" * 70)

print(model)

print()

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print(
    f"Total Parameters     : "
    f"{total_parameters:,}"
)

print(
    f"Trainable Parameters : "
    f"{trainable_parameters:,}"
)

print()


# =========================================================
# 10. Loss / Optimizer
# =========================================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)


print("[4] Training Configuration")
print("-" * 70)

print(
    "Loss      : CrossEntropyLoss"
)

print(
    "Optimizer : Adam"
)

print(
    f"LR        : {LEARNING_RATE}"
)

print()


# =========================================================
# 11. Train
# =========================================================

print("[5] Training")
print("-" * 70)

training_start = time.perf_counter()

(
    model,
    history,
    best_val_macro_f1,
    training_time
) = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=EPOCHS,
    device=device
)

training_end = time.perf_counter()

training_time = (
    training_end
    - training_start
)


print()

print(
    f"Best Validation Macro F1 : "
    f"{best_val_macro_f1:.4f}"
)

print(
    f"Training Time            : "
    f"{training_time:.2f} sec"
)

print()


# =========================================================
# 12. Save Best Model
# =========================================================

torch.save(
    model.state_dict(),
    MODEL_PATH
)

print(
    f"Model saved : "
    f"{MODEL_PATH}"
)

print()


# =========================================================
# 13. Test Evaluation
# =========================================================

print("[6] Test Evaluation")
print("-" * 70)

test_start = time.perf_counter()

(
    test_loss,
    y_true,
    y_pred
) = evaluate(
    model=model,
    dataloader=test_loader,
    criterion=criterion,
    device=device
)

test_end = time.perf_counter()

inference_time = (
    test_end
    - test_start
)


# =========================================================
# 14. Calculate Metrics
# =========================================================

metrics = calculate_metrics(
    y_true,
    y_pred
)

print()

print_metrics(
    metrics
)

print()

print(
    f"Test Loss          : "
    f"{test_loss:.4f}"
)

print(
    f"Test Inference Time: "
    f"{inference_time:.2f} sec"
)

print()


# =========================================================
# 15. Classification Report
# =========================================================

print("[7] Classification Report")
print("-" * 70)

report = get_classification_report(
    y_true,
    y_pred
)

print(report)


# =========================================================
# 16. Confusion Matrix
# =========================================================

print("[8] Confusion Matrix")
print("-" * 70)

cm = get_confusion_matrix(
    y_true,
    y_pred
)

print(cm)

print()


# =========================================================
# 17. Save Confusion Matrix
# =========================================================

plt.figure(
    figsize=(8, 7)
)

plt.imshow(
    cm,
    interpolation="nearest"
)

plt.title(
    "BoW + MLP Confusion Matrix"
)

plt.xlabel(
    "Predicted Label"
)

plt.ylabel(
    "True Label"
)

plt.xticks(
    range(NUM_CLASSES)
)

plt.yticks(
    range(NUM_CLASSES)
)

for i in range(NUM_CLASSES):

    for j in range(NUM_CLASSES):

        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()

plt.tight_layout()

plt.savefig(
    CONFUSION_PATH,
    dpi=200
)

plt.close()


print(
    f"Confusion matrix saved : "
    f"{CONFUSION_PATH}"
)

print()


# =========================================================
# 18. Training Curve
# =========================================================

CURVE_PATH = (
    Path("results")
    / "bow_training_curve.png"
)

plt.figure(
    figsize=(8, 5)
)

epochs_range = range(
    1,
    EPOCHS + 1
)

plt.plot(
    epochs_range,
    history["train_loss"],
    marker="o",
    label="Train Loss"
)

plt.plot(
    epochs_range,
    history["val_loss"],
    marker="o",
    label="Validation Loss"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)

plt.title(
    "BoW + MLP Training Curve"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    CURVE_PATH,
    dpi=200
)

plt.close()


print(
    f"Training curve saved : "
    f"{CURVE_PATH}"
)

print()


# =========================================================
# 19. Final Summary
# =========================================================

print("=" * 70)
print("BoW + MLP Baseline Complete")
print("=" * 70)

print()

print(
    f"Best Validation Macro F1 : "
    f"{best_val_macro_f1:.4f}"
)

print(
    f"Test Accuracy            : "
    f"{metrics['accuracy']:.4f}"
)

print(
    f"Test Macro Precision     : "
    f"{metrics['macro_precision']:.4f}"
)

print(
    f"Test Macro Recall        : "
    f"{metrics['macro_recall']:.4f}"
)

print(
    f"Test Macro F1            : "
    f"{metrics['macro_f1']:.4f}"
)

print(
    f"Test Weighted F1         : "
    f"{metrics['weighted_f1']:.4f}"
)

print()

print(
    f"Training Time            : "
    f"{training_time:.2f} sec"
)

print(
    f"Test Inference Time      : "
    f"{inference_time:.2f} sec"
)

print()

print("=" * 70)

04. Baseline - BoW + MLP

Device         : cuda
Max Features   : 1000
Hidden Dim     : 256
Batch Size     : 64
Epochs         : 10
Learning Rate  : 0.001

[1] Load Split Data
----------------------------------------------------------------------
Train      : 305,345
Validation : 65,422
Test       : 65,355

[2] Create Bag-of-Words Representation
----------------------------------------------------------------------
Vocabulary Size : 1,000
Train Matrix    : (305345, 1000)
Validation      : (65422, 1000)
Test Matrix     : (65355, 1000)
Train Sparsity  : 0.9872

Vectorizer saved : results/checkpoints/bow_vectorizer.pkl

[3] Model
----------------------------------------------------------------------
BoWMLP(
  (network): Sequential(
    (0): Linear(in_features=1000, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=6, bias=True)
  )
)

Total Parameters     : 257,798
Trainable Parameters : 257,798

[4] Training Configuration
--------------------------